# Tutorial: Poisoning e Backdoor

**Público:** red/blue teams de ML.

**Objetivos:** diferenciar label flipping de backdoor, preservar seeds e medir contaminação.

## Roteiro

1. Criar dados sintéticos.
2. Aplicar ataques pelo módulo de produção.
3. Inspecionar labels e trigger.
4. Exercício: comparar taxas.

> Pitfall: avaliar apenas accuracy limpa; backdoors exigem attack success rate no trigger.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts" / "python"))
import numpy as np
from sklearn.datasets import make_classification
from ml_security.poisoning_attacks import PoisoningAttackSimulator

np.random.seed(42)
X, y = make_classification(n_samples=200, n_features=6, random_state=42)
simulator = PoisoningAttackSimulator(log_dir=ROOT / "results" / "ml_security")

## Ataques

Label flipping altera rótulos. Backdoor injeta um trigger na última feature e redireciona o alvo.

In [2]:
_, y_flipped, flip_meta = simulator.label_flipping_indiscriminado(X, y, poison_rate=0.10)
X_backdoor, y_backdoor, backdoor_meta = simulator.backdoor_attack(
    X, y, backdoor_class=1, trigger_pattern=9.9, poison_rate=0.05
)
{
    "label_flipping": flip_meta,
    "backdoor": backdoor_meta,
    "labels_changed": int((y_flipped != y).sum()),
    "triggered_rows": int((X_backdoor[:, -1] == 9.9).sum()),
}

{'label_flipping': {'attack_type': 'label_flipping_indiscriminado',
  'poison_rate': 0.1,
  'n_poisoned': 20,
  'poison_indices': [95,
   15,
   30,
   158,
   128,
   115,
   69,
   170,
   174,
   45,
   66,
   182,
   165,
   78,
   186,
   177,
   56,
   152,
   82,
   68]},
 'backdoor': {'attack_type': 'backdoor',
  'backdoor_class': 1,
  'trigger_pattern': 9.9,
  'poison_rate': 0.05,
  'n_poisoned': 10,
  'poison_indices': [176, 33, 121, 180, 102, 135, 88, 76, 65, 124]},
 'labels_changed': 20,
 'triggered_rows': 10}

## Exercício

Repita com 1%, 5% e 10% de poison. Compare n_poisoned e documente clean accuracy, poisoned accuracy e backdoor ASR em um modelo treinado.

**Extensão:** use o módulo clean_label_poisoning.py para verificar label_integrity.

In [3]:
summary = []
for rate in [0.01, 0.05, 0.10]:
    _, poisoned_y, metadata = simulator.label_flipping_indiscriminado(X, y, rate)
    summary.append({"poison_fraction": rate, "labels_changed": int((poisoned_y != y).sum())})
summary

[{'poison_fraction': 0.01, 'labels_changed': 2},
 {'poison_fraction': 0.05, 'labels_changed': 10},
 {'poison_fraction': 0.1, 'labels_changed': 20}]